##Load the document

In [ ]:
!pip install -qU langchain-community pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "/content/drive/MyDrive/data sets/NIPS-2017-attention-is-all-you-need-Paper.pdf"
loader = PyPDFLoader(file_path)
doc = loader.load()
print(doc[0].metadata)

/tmp/ipykernel_28574/934918719.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


{'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On English-to-French t

##Text Spliter

In [ ]:
!pip install -qU langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap  = 200,
)
all_splits = text_splitter.split_documents(doc)
print(all_splits)
print(len(all_splits))

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

## Creating embbedings

In [ ]:
!pip install -qU langchain langchain-huggingface sentence_transformers

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
!pip install -qU langchain-chroma

In [ ]:
!pip install "opentelemetry-api==1.38.0" "opentelemetry-sdk==1.38.0" "opentelemetry-exporter-otlp-proto-grpc==1.38.0" --quiet

In [ ]:
from langchain_chroma import Chroma
vector_store = Chroma(
    collection_name = "NIPS-2017",
    embedding_function = embedding_model,
    persist_directory = "./chroma_langchain_db"
)

document_ids = vector_store.add_documents(documents = all_splits)
print(document_ids)
sample = vector_store.get(limit=1,include=["metadatas","documents"])
print(sample)

['450694ae-af4c-4caa-8442-9373ffed08ce', 'a5b9275e-68d9-429a-bff0-54e5e02b3ada', 'd924d90c-e8b9-47a6-bfe9-7fbd9ddd6b5f', 'c5078947-8597-48ba-9b6a-bcd0d0f914f6', '7ca6df84-3234-42e7-bf0e-293a2aae01cf', 'c6c698c2-abf6-4774-991d-f5c81b89ad29', '6575c725-9e1c-44c5-8815-32f3c508e2ca', '58fb9ae0-67d0-4359-9f35-ddc05f562a1c', '794ee162-d009-469f-a965-d396209caa9a', 'b3e786fe-7497-4b55-972b-2fce2b23bfe2', 'adbce4a1-0484-42d6-9983-f1b77c2c6940', '3a616b51-36d8-4967-8239-584c9cc25bcb', 'ae4d2725-66cc-4643-ae18-aa593c0b058f', '147882a9-fdfc-4e9d-ade3-e962895727ad', '8b1f23fd-147c-46cd-afc7-1885f9124a97', '47a0d5e6-e04d-4515-a088-74a2621b1d65', 'cefb2578-4585-4dbb-b197-93ca66b3ff28', '42b6bd15-73d3-42bb-af4e-ecad6da535eb', 'c5ecee76-645b-4d33-aa52-b5a6a5955472', 'cde4e7f2-7f6b-410c-8741-f9f8482aafc9', '4360f593-a8ac-410d-851d-1c34ce6b29cb', 'e853827c-2732-4d5d-ac6a-3815f0c38a66', '10d7403d-62f9-4347-871c-1caeb85a539c', '2ddef416-8efe-43e8-92cf-d1f57e1d4a0a', '1121b228-944c-4122-8be2-20ed93dee2ba',

##retrive and generate

In [ ]:
def retrive_context(query:str,k:int=2):
    retrived_docs = vector_store.similarity_search(query,k=k)
    docs_content = ""
    for doc in retrived_docs:
        docs_content += f"Sourse: {doc.metadata['source']}\n"
        docs_content += f"Content: {doc.page_content}\n"
    return docs_content, retrived_docs

## Model

In [ ]:
!pip install -U langchain-google-genai

In [ ]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

api_key = userdata.get("gemini_api_key")
model = init_chat_model(
    "google_genai:gemini-2.5-flash",
    api_key = api_key,
)


In [ ]:
def docu_chat(user_query):
    context, docs = retrive_context(user_query,k=2)
    system_message = f"""You are a helpful chatbot.
                        Use only the following pices of context to answer the
                        question. Don't makeup any new information: {context}"""

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_query}
    ]
    response = model.invoke(messages)
    return {
        "answer":response.content,
        "source_documents":docs,
        "context_used": context
    }

In [ ]:
result = docu_chat("explain what is the use of decoders in transformers")
print(result["answer"])

NameError: name 'ask_about_pdf' is not defined